<a href="https://colab.research.google.com/github/enriquefroes/treinadores-brasileirao-2026/blob/main/notebooks/00_dados_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import pandas as pd
from io import StringIO

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/analise_treinadores_brasileirao'
except ImportError:
    BASE = './analise_treinadores_brasileirao'

DADOS = os.path.join(BASE, 'dados')
for p in ['dados', 'graficos', 'tabelas']:
    os.makedirs(os.path.join(BASE, p), exist_ok=True)
print('Pastas prontas em', BASE)

Mounted at /content/drive
Pastas prontas em /content/drive/MyDrive/analise_treinadores_brasileirao


In [2]:
SOBRESCREVER = False

ARQUIVOS = {
    'treinadores.csv': """clube,treinador,tipo,inicio,fim,precisao,obs
Palmeiras,Abel Ferreira,efetivo,2020-11-04,,exata,
Bahia,Rogério Ceni,efetivo,2023-09-09,,exata,
Mirassol,Rafael Guanaes,efetivo,2025-03-18,,exata,
Athletico-PR,Odair Hellmann,efetivo,2025-05-22,,exata,
Vitória,Jair Ventura,efetivo,2025-09-24,,exata,
Bragantino,Vagner Mancini,efetivo,2025-10-30,,exata,
Coritiba,Fernando Seabra,efetivo,2025-12-08,,exata,
Atlético-MG,Jorge Sampaoli,efetivo,,2026-02-12,exata,início anterior a 2026
Atlético-MG,Lucas Gonçalves,interino,2026-02-13,2026-02-25,exata,
Atlético-MG,Eduardo Domínguez,efetivo,2026-02-26,,exata,
Vasco,Fernando Diniz,efetivo,,2026-02-22,exata,início anterior a 2026
Vasco,Bruno Lazaroni,interino,2026-02-23,2026-03-02,exata,1 jogo no Brasileirão: derrota para o Santos
Vasco,Renato Gaúcho,efetivo,2026-03-03,2026-06-18,exata,
Vasco,Pedro Emanuel,efetivo,2026-07-10,,exata,período sem treinador coincidiu com a pausa da Copa do Mundo
Remo,Juan Carlos Osorio,efetivo,,2026-03-01,exata,início anterior a 2026
Remo,Léo Condé,efetivo,2026-03-02,2026-09-15,início aproximado,"início em março, após a saída de Osorio"
Remo,Thiago Carpini,efetivo,2026-09-16,,exata,
Flamengo,Filipe Luís,efetivo,,2026-03-03,exata,início anterior a 2026
Flamengo,Leonardo Jardim,efetivo,2026-03-04,,exata,
São Paulo,Hernán Crespo,efetivo,,2026-03-09,exata,início anterior a 2026
São Paulo,Roger Machado,efetivo,2026-03-10,2026-05-13,exata,
São Paulo,Dorival Júnior,efetivo,2026-05-15,,exata,
Cruzeiro,Tite,efetivo,,2026-03-15,exata,início anterior a 2026
Cruzeiro,Wesley Carvalho,interino,2026-03-16,2026-03-22,exata,2 jogos no Brasileirão (1 empate e 1 derrota)
Cruzeiro,Artur Jorge,efetivo,2026-03-23,,exata,
Santos,Juan Pablo Vojvoda,efetivo,,2026-03-18,exata,início anterior a 2026
Santos,Cuca,efetivo,2026-03-19,,exata,
Botafogo,Martín Anselmi,efetivo,,2026-03-22,exata,início anterior a 2026
Botafogo,Belão,interino,2026-03-23,2026-04-01,exata,1ª passagem como interino no Brasileirão
Botafogo,Franclim Carvalho,efetivo,2026-04-02,2026-08-18,exata,
Botafogo,Belão,interino,2026-08-19,2026-09-16,exata,2ª passagem como interino no Brasileirão
Botafogo,Tite,efetivo,2026-09-17,,exata,
Chapecoense,Gilmar Dal Pozzo,efetivo,,2026-04-03,exata,início anterior a 2026
Chapecoense,Fábio Matias,efetivo,2026-04-05,2026-05-25,exata,
Chapecoense,Rafael Lacerda,efetivo,2026-05-30,,exata,
Corinthians,Dorival Júnior,efetivo,,2026-04-05,exata,início anterior a 2026
Corinthians,Fernando Diniz,efetivo,2026-04-06,,exata,
Fluminense,Luis Zubeldía,efetivo,,2026-08-13,exata,início anterior a 2026
Fluminense,Marcão,efetivo,2026-08-14,,exata,
Grêmio,Luís Castro,efetivo,2025-12-01,2026-09-13,exata,início em dezembro de 2025
Grêmio,Renato Gaúcho,efetivo,2026-09-14,,exata,
Internacional,Paulo Pezzolano,efetivo,,2026-09-20,exata,início anterior a 2026
Internacional,Eduardo Baptista,efetivo,2026-09-24,,exata,
""",
    'partidas.csv': """rodada,data,mandante,gols_mandante,gols_visitante,visitante
1,2026-01-28,Atlético-MG,2,2,Palmeiras
1,2026-01-28,Internacional,0,1,Athletico-PR
1,2026-01-28,Coritiba,0,1,Bragantino
1,2026-01-28,Vitória,2,0,Remo
1,2026-01-28,Fluminense,2,1,Grêmio
1,2026-01-28,Corinthians,1,2,Bahia
1,2026-01-28,Chapecoense,4,2,Santos
1,2026-01-28,São Paulo,2,1,Flamengo
1,2026-01-29,Mirassol,2,1,Vasco
1,2026-01-29,Botafogo,4,0,Cruzeiro
2,2026-02-04,Flamengo,1,1,Internacional
2,2026-02-04,Bragantino,1,0,Atlético-MG
2,2026-02-04,Santos,1,1,São Paulo
2,2026-02-04,Remo,2,2,Mirassol
2,2026-02-04,Palmeiras,5,1,Vitória
2,2026-02-04,Grêmio,5,3,Botafogo
2,2026-02-05,Bahia,1,1,Fluminense
2,2026-02-05,Vasco,1,1,Chapecoense
2,2026-02-05,Cruzeiro,1,2,Coritiba
3,2026-02-10,Vitória,1,2,Flamengo
3,2026-02-11,Mirassol,2,2,Cruzeiro
3,2026-02-11,Chapecoense,3,3,Coritiba
3,2026-02-11,Atlético-MG,3,3,Remo
3,2026-02-11,Vasco,0,1,Bahia
3,2026-02-11,São Paulo,2,0,Grêmio
3,2026-02-12,Athletico-PR,2,1,Santos
3,2026-02-12,Fluminense,1,0,Botafogo
3,2026-02-12,Corinthians,2,0,Bragantino
3,2026-02-12,Internacional,1,3,Palmeiras
2,2026-02-19,Athletico-PR,0,1,Corinthians
4,2026-02-25,Bragantino,1,1,Athletico-PR
4,2026-02-25,Remo,1,1,Internacional
4,2026-02-25,Coritiba,0,1,São Paulo
4,2026-02-25,Cruzeiro,1,1,Corinthians
4,2026-02-25,Palmeiras,2,1,Fluminense
4,2026-02-25,Grêmio,2,1,Atlético-MG
4,2026-02-26,Santos,2,1,Vasco
5,2026-03-10,Mirassol,2,2,Santos
5,2026-03-11,Atlético-MG,1,0,Internacional
5,2026-03-11,Bahia,1,1,Vitória
5,2026-03-11,Flamengo,2,0,Cruzeiro
5,2026-03-11,Corinthians,0,2,Coritiba
5,2026-03-12,Remo,0,2,Fluminense
5,2026-03-12,Vasco,2,1,Palmeiras
5,2026-03-12,São Paulo,2,0,Chapecoense
5,2026-03-12,Grêmio,1,1,Bragantino
6,2026-03-14,Vitória,2,0,Atlético-MG
6,2026-03-14,Botafogo,0,3,Flamengo
6,2026-03-15,Fluminense,3,2,Athletico-PR
6,2026-03-15,Santos,1,1,Corinthians
6,2026-03-15,Internacional,0,1,Bahia
6,2026-03-15,Palmeiras,1,0,Mirassol
6,2026-03-15,Coritiba,1,0,Remo
6,2026-03-15,Bragantino,1,2,São Paulo
6,2026-03-15,Cruzeiro,3,3,Vasco
6,2026-03-16,Chapecoense,1,1,Grêmio
7,2026-03-18,Palmeiras,2,1,Botafogo
7,2026-03-18,Bahia,2,0,Bragantino
7,2026-03-18,Athletico-PR,2,1,Cruzeiro
7,2026-03-18,Mirassol,0,1,Coritiba
7,2026-03-18,Atlético-MG,1,0,São Paulo
7,2026-03-18,Vasco,3,2,Fluminense
7,2026-03-18,Santos,1,2,Internacional
7,2026-03-18,Grêmio,2,0,Vitória
7,2026-03-19,Flamengo,3,0,Remo
7,2026-03-19,Chapecoense,0,0,Corinthians
8,2026-03-21,Bragantino,1,2,Botafogo
8,2026-03-21,Fluminense,1,0,Atlético-MG
8,2026-03-21,São Paulo,0,1,Palmeiras
8,2026-03-22,Vasco,2,1,Grêmio
8,2026-03-22,Cruzeiro,0,0,Santos
8,2026-03-22,Athletico-PR,2,0,Coritiba
8,2026-03-22,Remo,4,1,Bahia
8,2026-03-22,Internacional,2,0,Chapecoense
8,2026-03-22,Vitória,1,0,Mirassol
8,2026-03-22,Corinthians,1,1,Flamengo
5,2026-03-29,Athletico-PR,4,1,Botafogo
9,2026-04-01,Botafogo,3,2,Mirassol
9,2026-04-01,Internacional,1,1,São Paulo
9,2026-04-01,Cruzeiro,3,0,Vitória
9,2026-04-01,Bahia,3,0,Athletico-PR
9,2026-04-01,Coritiba,1,1,Vasco
9,2026-04-01,Fluminense,3,1,Corinthians
9,2026-04-02,Santos,2,0,Remo
9,2026-04-02,Chapecoense,0,4,Atlético-MG
9,2026-04-02,Palmeiras,2,1,Grêmio
9,2026-04-02,Bragantino,3,0,Flamengo
10,2026-04-04,São Paulo,4,1,Cruzeiro
10,2026-04-04,Coritiba,1,1,Fluminense
10,2026-04-04,Vasco,1,2,Botafogo
10,2026-04-05,Chapecoense,1,1,Vitória
10,2026-04-05,Flamengo,3,1,Santos
10,2026-04-05,Atlético-MG,2,1,Athletico-PR
10,2026-04-05,Corinthians,0,1,Internacional
10,2026-04-05,Bahia,1,2,Palmeiras
10,2026-04-05,Mirassol,0,1,Bragantino
10,2026-04-05,Grêmio,0,0,Remo
11,2026-04-11,Vitória,2,0,São Paulo
11,2026-04-11,Remo,1,1,Vasco
11,2026-04-11,Mirassol,1,2,Bahia
11,2026-04-11,Santos,1,0,Atlético-MG
11,2026-04-11,Internacional,0,0,Grêmio
11,2026-04-12,Athletico-PR,2,0,Chapecoense
11,2026-04-12,Botafogo,2,2,Coritiba
11,2026-04-12,Fluminense,1,2,Flamengo
11,2026-04-12,Corinthians,0,0,Palmeiras
11,2026-04-12,Cruzeiro,2,1,Bragantino
12,2026-04-18,Vasco,2,1,São Paulo
12,2026-04-18,Chapecoense,1,4,Botafogo
12,2026-04-18,Vitória,0,0,Corinthians
12,2026-04-18,Cruzeiro,2,0,Grêmio
12,2026-04-19,Internacional,1,2,Mirassol
12,2026-04-19,Santos,2,3,Fluminense
12,2026-04-19,Coritiba,2,0,Atlético-MG
12,2026-04-19,Palmeiras,1,0,Athletico-PR
12,2026-04-19,Bragantino,4,2,Remo
12,2026-04-19,Flamengo,2,0,Bahia
13,2026-04-25,Botafogo,2,2,Internacional
13,2026-04-25,Bahia,2,2,Santos
13,2026-04-25,Remo,0,1,Cruzeiro
13,2026-04-25,São Paulo,1,0,Mirassol
13,2026-04-26,Corinthians,1,0,Vasco
13,2026-04-26,Grêmio,1,0,Coritiba
13,2026-04-26,Bragantino,0,1,Palmeiras
13,2026-04-26,Athletico-PR,3,1,Vitória
13,2026-04-26,Fluminense,2,1,Chapecoense
13,2026-04-26,Atlético-MG,0,4,Flamengo
14,2026-05-02,Botafogo,1,2,Remo
14,2026-05-02,Palmeiras,1,1,Santos
14,2026-05-02,Vitória,4,1,Coritiba
14,2026-05-02,Athletico-PR,0,0,Grêmio
14,2026-05-02,Cruzeiro,1,3,Atlético-MG
14,2026-05-03,Flamengo,2,2,Vasco
14,2026-05-03,São Paulo,2,2,Bahia
14,2026-05-03,Internacional,2,0,Fluminense
14,2026-05-03,Chapecoense,1,2,Bragantino
14,2026-05-03,Mirassol,2,1,Corinthians
15,2026-05-09,Coritiba,2,2,Internacional
15,2026-05-09,Fluminense,2,2,Vitória
15,2026-05-09,Bahia,1,2,Cruzeiro
15,2026-05-10,Atlético-MG,1,1,Botafogo
15,2026-05-10,Remo,1,1,Palmeiras
15,2026-05-10,Santos,2,0,Bragantino
15,2026-05-10,Corinthians,3,2,São Paulo
15,2026-05-10,Mirassol,1,1,Chapecoense
15,2026-05-10,Grêmio,0,1,Flamengo
15,2026-05-10,Vasco,1,0,Athletico-PR
16,2026-05-16,Atlético-MG,3,1,Mirassol
16,2026-05-16,Internacional,4,1,Vasco
16,2026-05-16,Fluminense,2,1,São Paulo
16,2026-05-16,Palmeiras,1,1,Cruzeiro
16,2026-05-17,Santos,0,3,Coritiba
16,2026-05-17,Botafogo,3,1,Corinthians
16,2026-05-17,Bahia,1,1,Grêmio
16,2026-05-17,Bragantino,2,0,Vitória
16,2026-05-17,Chapecoense,2,3,Remo
16,2026-05-17,Athletico-PR,1,1,Flamengo
17,2026-05-23,São Paulo,1,1,Botafogo
17,2026-05-23,Vitória,2,0,Internacional
17,2026-05-23,Mirassol,1,0,Fluminense
17,2026-05-23,Grêmio,3,2,Santos
17,2026-05-23,Flamengo,0,3,Palmeiras
17,2026-05-24,Cruzeiro,2,1,Chapecoense
17,2026-05-24,Remo,1,2,Athletico-PR
17,2026-05-24,Corinthians,1,0,Atlético-MG
17,2026-05-24,Vasco,0,3,Bragantino
17,2026-05-25,Coritiba,3,2,Bahia
18,2026-05-30,Flamengo,3,0,Coritiba
18,2026-05-30,Athletico-PR,1,0,Mirassol
18,2026-05-30,Grêmio,1,3,Corinthians
18,2026-05-30,Bahia,2,1,Botafogo
18,2026-05-30,Santos,3,1,Vitória
18,2026-05-31,Bragantino,3,1,Internacional
18,2026-05-31,Vasco,0,1,Atlético-MG
18,2026-05-31,Palmeiras,1,0,Chapecoense
18,2026-05-31,Cruzeiro,1,1,Fluminense
18,2026-05-31,Remo,1,0,São Paulo
19,2026-07-16,Botafogo,2,1,Santos
19,2026-07-16,Vitória,1,0,Vasco
4,2026-07-17,Bahia,2,0,Chapecoense
19,2026-07-17,Fluminense,1,1,Bragantino
19,2026-07-17,Mirassol,2,1,Grêmio
19,2026-07-21,Atlético-MG,1,1,Bahia
19,2026-07-22,Coritiba,1,3,Palmeiras
19,2026-07-22,São Paulo,1,2,Athletico-PR
19,2026-07-22,Internacional,1,2,Cruzeiro
19,2026-07-22,Chapecoense,0,4,Flamengo
4,2026-07-23,Botafogo,0,0,Vitória
19,2026-07-23,Corinthians,3,0,Remo
20,2026-07-25,Athletico-PR,2,0,Internacional
20,2026-07-25,Santos,2,2,Chapecoense
20,2026-07-25,Vasco,1,1,Mirassol
20,2026-07-26,Bahia,1,1,Corinthians
20,2026-07-26,Cruzeiro,0,1,Botafogo
20,2026-07-26,Grêmio,1,1,Fluminense
20,2026-07-26,Bragantino,0,0,Coritiba
20,2026-07-26,Flamengo,1,1,São Paulo
20,2026-07-26,Remo,2,0,Vitória
20,2026-07-26,Palmeiras,1,2,Atlético-MG
21,2026-07-29,Internacional,1,1,Flamengo
21,2026-07-29,Mirassol,2,1,Remo
21,2026-07-29,Vitória,0,4,Palmeiras
21,2026-07-29,Fluminense,0,0,Bahia
21,2026-07-30,Corinthians,0,0,Athletico-PR
21,2026-07-30,Coritiba,0,1,Cruzeiro
22,2026-08-08,Grêmio,2,1,São Paulo
22,2026-08-08,Remo,2,2,Atlético-MG
22,2026-08-08,Coritiba,2,1,Chapecoense
22,2026-08-08,Botafogo,1,1,Fluminense
22,2026-08-09,Cruzeiro,3,1,Mirassol
22,2026-08-09,Bahia,0,0,Vasco
22,2026-08-09,Palmeiras,0,0,Internacional
22,2026-08-09,Bragantino,0,2,Corinthians
22,2026-08-09,Santos,0,2,Athletico-PR
22,2026-08-09,Flamengo,2,0,Vitória
23,2026-08-15,Fluminense,3,2,Palmeiras
23,2026-08-15,Athletico-PR,1,1,Bragantino
23,2026-08-15,São Paulo,1,1,Coritiba
23,2026-08-16,Chapecoense,3,3,Bahia
23,2026-08-16,Atlético-MG,3,0,Grêmio
23,2026-08-16,Vasco,0,3,Santos
23,2026-08-16,Vitória,1,0,Botafogo
23,2026-08-16,Mirassol,1,5,Flamengo
23,2026-08-16,Corinthians,1,2,Cruzeiro
23,2026-08-17,Internacional,1,1,Remo
24,2026-08-22,Fluminense,2,1,Remo
24,2026-08-22,Internacional,0,0,Atlético-MG
24,2026-08-22,Cruzeiro,2,1,Flamengo
24,2026-08-23,Vitória,0,2,Bahia
24,2026-08-23,Palmeiras,4,1,Vasco
24,2026-08-23,Bragantino,1,0,Grêmio
24,2026-08-23,Chapecoense,1,0,São Paulo
24,2026-08-23,Santos,1,1,Mirassol
24,2026-08-23,Coritiba,2,1,Corinthians
24,2026-08-24,Botafogo,2,3,Athletico-PR
25,2026-08-29,Atlético-MG,2,1,Vitória
25,2026-08-29,São Paulo,2,1,Bragantino
25,2026-08-29,Vasco,3,1,Cruzeiro
25,2026-08-30,Athletico-PR,3,3,Fluminense
25,2026-08-30,Flamengo,3,0,Botafogo
25,2026-08-30,Corinthians,0,1,Santos
25,2026-08-30,Grêmio,3,1,Chapecoense
25,2026-08-30,Mirassol,1,1,Palmeiras
25,2026-08-30,Bahia,3,2,Internacional
25,2026-08-31,Remo,2,3,Coritiba
4,2026-09-02,Flamengo,2,0,Mirassol
26,2026-09-05,Bragantino,2,3,Bahia
26,2026-09-05,São Paulo,2,0,Atlético-MG
26,2026-09-05,Fluminense,1,0,Vasco
26,2026-09-06,Coritiba,1,2,Mirassol
26,2026-09-06,Remo,0,1,Flamengo
26,2026-09-06,Internacional,2,3,Santos
26,2026-09-06,Cruzeiro,3,1,Athletico-PR
26,2026-09-06,Botafogo,0,0,Palmeiras
26,2026-09-06,Corinthians,1,2,Chapecoense
26,2026-09-07,Vitória,1,0,Grêmio
27,2026-09-11,Coritiba,3,3,Athletico-PR
27,2026-09-12,Grêmio,1,2,Vasco
27,2026-09-12,Atlético-MG,3,1,Fluminense
27,2026-09-12,Chapecoense,1,2,Internacional
27,2026-09-12,Palmeiras,2,0,São Paulo
27,2026-09-12,Botafogo,1,1,Bragantino
27,2026-09-12,Santos,2,1,Cruzeiro
27,2026-09-13,Mirassol,2,2,Vitória
27,2026-09-13,Flamengo,2,1,Corinthians
27,2026-09-14,Bahia,2,1,Remo
21,2026-09-16,Botafogo,3,2,Grêmio
28,2026-09-19,Atlético-MG,1,1,Chapecoense
28,2026-09-19,Mirassol,2,0,Botafogo
28,2026-09-19,Remo,1,2,Santos
28,2026-09-19,Vasco,5,0,Coritiba
28,2026-09-19,São Paulo,2,0,Internacional
28,2026-09-20,Grêmio,0,0,Palmeiras
28,2026-09-20,Vitória,1,3,Cruzeiro
28,2026-09-20,Corinthians,1,3,Fluminense
28,2026-09-20,Flamengo,2,1,Bragantino
28,2026-09-20,Athletico-PR,2,1,Bahia
""",
    'excecoes.csv': """data,clube,treinador,tipo,motivo
2026-04-04,Botafogo,Belão,interino,"Franclim Carvalho já contratado e presente, mas Belão comandou da beira do campo"
""",
    'folha.csv': """clube,folha_mensal
Flamengo,53
Palmeiras,45
Corinthians,44
Fluminense,40
Cruzeiro,35
São Paulo,31
Atlético-MG,29
Santos,28
Botafogo,26
Bahia,26
Vasco,16.5
Internacional,15
Grêmio,13.5
Athletico-PR,9
Coritiba,8.5
Bragantino,7.7
Mirassol,5.5
Vitória,3.75
Remo,3.25
Chapecoense,2.25
""",
}
for nome, conteudo in ARQUIVOS.items():
    caminho = os.path.join(DADOS, nome)
    if SOBRESCREVER or not os.path.exists(caminho):
        pd.read_csv(StringIO(conteudo)).to_csv(caminho, index=False)
        print('Criado:', nome)
    else:
        print('Mantido:', nome)

Mantido: treinadores.csv
Mantido: partidas.csv
Mantido: excecoes.csv
Mantido: folha.csv


In [3]:
treinadores = pd.read_csv(os.path.join(DADOS, 'treinadores.csv'))
partidas = pd.read_csv(os.path.join(DADOS, 'partidas.csv'), parse_dates=['data'])
excecoes = pd.read_csv(os.path.join(DADOS, 'excecoes.csv'), parse_dates=['data'])
treinadores['ini'] = pd.to_datetime(treinadores['inicio']).fillna(pd.Timestamp('2000-01-01'))
treinadores['fim_'] = pd.to_datetime(treinadores['fim']).fillna(pd.Timestamp('2100-01-01'))

linhas = []
for _, p in partidas.iterrows():
    for mando, clube, adv, gp, gc in [('casa', p['mandante'], p['visitante'], p['gols_mandante'], p['gols_visitante']),
                                      ('fora', p['visitante'], p['mandante'], p['gols_visitante'], p['gols_mandante'])]:
        t = treinadores[(treinadores['clube'] == clube) & (treinadores['ini'] <= p['data']) & (treinadores['fim_'] >= p['data'])]
        if len(t) != 1:
            raise ValueError(f"{clube} em {p['data'].date()}: {len(t)} treinadores encontrados. Confira treinadores.csv")
        ex = excecoes[(excecoes['data'] == p['data']) & (excecoes['clube'] == clube)]
        treinador, tipo = (ex.iloc[0]['treinador'], ex.iloc[0]['tipo']) if len(ex) else (t.iloc[0]['treinador'], t.iloc[0]['tipo'])
        linhas.append({'rodada': p['rodada'], 'data': p['data'].date(), 'clube': clube, 'treinador': treinador, 'tipo': tipo,
                       'mando': mando, 'adversario': adv, 'gols_pro': gp, 'gols_contra': gc,
                       'pontos': 3 if gp > gc else 1 if gp == gc else 0})
jogos = pd.DataFrame(linhas).sort_values(['clube', 'data'])
jogos.to_csv(os.path.join(DADOS, 'jogos_por_treinador.csv'), index=False)
print(f"{len(partidas)} partidas · {len(jogos)} jogos atribuídos · {jogos.groupby(['clube','treinador']).ngroups} passagens")


tab = jogos.groupby('clube').agg(jogos=('pontos', 'size'), pontos=('pontos', 'sum')).sort_values('pontos', ascending=False)
tab

277 partidas · 554 jogos atribuídos · 41 passagens


,jogos,pontos
clube,,
Flamengo,28,60
Palmeiras,28,57
Athletico-PR,28,49
Fluminense,28,48
Bahia,28,46
Cruzeiro,28,45
Atlético-MG,27,40
Santos,27,38
Coritiba,28,38
